# v15b — chunked extraction

v14b took **one** hidden state per row: `h[:, -1, :]`, the last token. That single vector
carries the whole document, which is why a tool-output carrier and a one-line question land in
completely different regions of the space and cannot share a threshold.

v15b takes the hidden state at the **end of every chunk** of the document — from the *same*
forward pass. No extra model calls: the states were always there, v14b just threw them away.

## Why this should fix the threshold problem

Because the model is causal, the state at chunk *k* encodes chunks 1..k. So the **first
difference** `D_k = Z_k − Z_(k−1)` isolates what chunk *k* contributed, independent of where
the document started. A statistic like `max(D) − median(D)` asks:

> does **one part** of this document behave unlike the rest of it?

That is scale-free with respect to the document's baseline elevation — which is exactly the
quantity that spanned four orders of magnitude in v14.

It is also a direct extension of the thesis's own idea: v14 tracks trajectory across **depth**,
v15 tracks it across **depth and position**.

## Cost

Same number of forward passes as v14b. Extra cost is disk, not time. Only the indirect
families plus a PCA baseline are extracted — the direct families already meet target
(SPML ASR 1.24%, DeepSet 7.98%) and do not need chunk features.

In [ ]:
import os, json, math, time, gc
import numpy as np
import pandas as pd

MODEL_ID    = "meta-llama/Meta-Llama-3-8B-Instruct"
HOOK_LAYERS = [4, 8, 12, 16, 20, 24, 28, 31]
SYSTEM_PROMPT = ("You are a helpful, secure, and harmless AI assistant. "
                 "You must refuse to assist with any dangerous, illegal, "
                 "or unethical requests.")

MAX_LEN      = 512     # same cap as v14b
# PROPORTIONAL chunking, not a fixed stride. A fixed 32-token stride was measured against
# the v15a corpus and would have gutted the version: InjecAgent documents run ~98 tokens,
# giving a MEDIAN OF 3 CHUNKS -> 2 deltas, and `max - median` over two values is
# degenerate. 43-53% of InjecAgent rows fell below 3 chunks. That is the family with
# ASR 19.35% that v15 exists to fix, so the statistic would have been dead exactly where
# it is needed. BIPIA (6.7 median) was fine, which is how it nearly slipped through.
#
# Instead every row is cut into the SAME number of positions, spaced proportionally over
# its real tokens, with a floor on chunk size so very short rows do not get slivers:
#   n_chunks = clamp(n_tok // MIN_CHUNK_TOK, 2, MAX_CHUNKS)
# InjecAgent (~98 tok) -> 12 chunks; BIPIA (~215 tok) -> 16 chunks of ~13 tok;
# a full 512-token row -> 16 chunks of 32 tok, identical to the old fixed stride.
#
# Side benefit: order statistics become directly comparable across documents, because
# every row contributes the same number of deltas, and argmax/K is a true RELATIVE
# position rather than an absolute token offset.
# Cost and storage are unchanged -- MAX_CHUNKS still bounds the array.
MAX_CHUNKS    = 16
MIN_CHUNK_TOK = 8      # never cut below this; a sliver is noise, not a segment
MIN_CHUNKS    = 3      # rows still below this get no position statistics

# 24 GB budget: identical to v14b, which completed 11,838 rows without wedging.
TOKEN_BUDGET, MAX_BATCH, DEEP_CLEAN_EVERY = 1024, 4, 25
SHARD = 250            # smaller than v14b: each row is MAX_CHUNKS x bigger on disk
DTYPE_OUT = np.float16 # fp16 halves 4 GB -> 2 GB; PCA runs in float32 downstream

CORPUS = "datasets/unified/injection_corpus_v15.csv"
OUTDIR = "representation_cache/chunked_v15"
os.makedirs(OUTDIR, exist_ok=True)

# Only the indirect families need chunk features. The direct families already meet
# target under v14 features and are read from the v14 cache by v15c.
EXTRACT = ["BIPIA", "BIPIA-Clean", "BIPIA-HardNeg",
           "InjecAgent", "InjecAgent-Clean", "InjecAgent-HardNeg"]
BASELINE_N = 400       # Dolly rows, to fit PCA in CHUNK space rather than reusing
                       # v14's basis (which was fitted on last-token states only)

full = pd.read_csv(CORPUS).reset_index(drop=True)
full["_src"] = np.arange(len(full))
sel = full[full.corpus.isin(EXTRACT)].copy()
base = full[full.family == "baseline"].sample(
    n=min(BASELINE_N, int((full.family == "baseline").sum())), random_state=42).copy()
base["corpus"] = "Baseline-Dolly"
df = pd.concat([sel, base], ignore_index=True)
texts = df.text.astype(str).tolist()
N = len(texts)
n_shards = math.ceil(N / SHARD)

done = {int(f.split("_")[1].split(".")[0]) for f in os.listdir(OUTDIR) if f.startswith("shard_")}
todo = [i for i in range(n_shards) if i not in done]

gb = N * len(HOOK_LAYERS) * MAX_CHUNKS * 4096 * 2 / 1e9
print(df.groupby("corpus").size().to_string())
print(f"\nrows        : {N:,}   ({df.label.sum():,} injections)")
print(f"chunks      : proportional, up to {MAX_CHUNKS} positions/row, >={MIN_CHUNK_TOK} tok each")
_est = (df.n_chars / 4).clip(upper=MAX_LEN)
_nch = (_est // MIN_CHUNK_TOK).clip(2, MAX_CHUNKS)
print(f"  predicted chunks/row by corpus:")
for _c in df.corpus.unique():
    _x = _nch[df.corpus.values == _c]
    print(f"    {_c:22s} med {_x.median():4.1f}  p10 {_x.quantile(.1):4.1f}  "
          f"<{MIN_CHUNKS}: {(_x < MIN_CHUNKS).mean()*100:4.1f}%")
print(f"shards      : {n_shards} x {SHARD}   done {len(done)}   remaining {len(todo)}")
print(f"disk        : ~{gb:.1f} GB at fp16")
print(f"ETA         : ~{N/0.5/60:.0f} min at v14b's measured ~0.5 rows/s")
print("\nIf that is too long, lower BIPIA_PER_PAYLOAD in v15a and rebuild -- do NOT")
print("reduce MAX_LEN here, it would change the chunk grid and break comparability.")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ.get("HF_TOKEN"))
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "left"          # real tokens occupy the LAST L positions

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, token=os.environ.get("HF_TOKEN"))
DEV = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(DEV).eval()
base_m = model.model

# GATHER holds, per batch, the padded-tensor index of each chunk end: (B, MAX_CHUNKS).
# The hook gathers only those positions, so device memory per layer is
# B x MAX_CHUNKS x 4096 (~0.5 MB) instead of the full B x T x 4096.
GATHER = None
BUF = {}
def _mk(li):
    def hook(_m, _i, out):
        h = out[0] if isinstance(out, tuple) else out          # (B, T, H)
        idx = GATHER[:, :, None].expand(-1, -1, h.shape[-1])   # (B, MC, H)
        BUF[li] = torch.take_along_dim(h, idx, dim=1).detach().to(torch.float16).cpu()
    return hook
HANDLES = [base_m.layers[li].register_forward_hook(_mk(li)) for li in HOOK_LAYERS]

def fmt(p):
    return tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": p}],
        tokenize=False, add_generation_prompt=True)

print(f"model on {DEV} | {len(HANDLES)} hooks | gather {MAX_CHUNKS} positions/layer")
if DEV == "mps":
    print(f"  MPS allocated {torch.mps.current_allocated_memory()/1e9:.2f} GB"
          f" | driver {torch.mps.driver_allocated_memory()/1e9:.2f} GB")
if DEV != "mps":
    print("\n  *** NOT ON MPS -- extraction will be ~20x slower. Fix torch first. ***")

In [ ]:
def chunk_index(attn):
    # attn: (B, T) attention mask, LEFT padded. Returns (B, MAX_CHUNKS) padded-tensor
    # indices of each chunk end, plus (B,) count of VALID chunks.
    # Chunks are spaced PROPORTIONALLY over each row's real tokens, so a short document
    # gets the same number of positions as a long one (see the config cell for why).
    B, T = attn.shape
    L = attn.sum(1)                                  # real token count per row
    idx = torch.zeros(B, MAX_CHUNKS, dtype=torch.long)
    nch = torch.zeros(B, dtype=torch.long)
    for b in range(B):
        Lb = int(L[b]); off = T - Lb                 # real tokens live at [off, T)
        k = int(min(MAX_CHUNKS, max(2, Lb // MIN_CHUNK_TOK)))
        # ends[j] is the last token of segment j; the final segment always closes on the
        # last real token, so the deepest state is the whole-document state (== v14).
        ends = [min(Lb - 1, int(round((j + 1) * Lb / k)) - 1) for j in range(k)]
        ends = sorted(set(e for e in ends if e >= 0))
        if not ends: ends = [max(0, Lb - 1)]
        if ends[-1] != Lb - 1: ends.append(Lb - 1)
        ends = ends[:MAX_CHUNKS]
        nch[b] = len(ends)
        for c in range(MAX_CHUNKS):
            idx[b, c] = off + (ends[c] if c < len(ends) else ends[-1])
    return idx.to(DEV), nch

def _ntok(t): return min(MAX_LEN, max(8, len(t) // 4 + 8))

def _batches(order, tl):
    cur, cmax = [], 0
    for j in order:
        n = _ntok(tl[j]); m = max(cmax, n)
        if cur and (m * (len(cur) + 1) > TOKEN_BUDGET or len(cur) >= MAX_BATCH):
            yield cur; cur, cmax = [j], n
        else:
            cur.append(j); cmax = m
    if cur: yield cur

def run_shard(si):
    global GATHER, MAX_LEN
    lo, hi = si * SHARD, min((si + 1) * SHARD, N)
    local = texts[lo:hi]
    order = sorted(range(len(local)), key=lambda j: _ntok(local[j]))
    out = {li: np.zeros((len(local), MAX_CHUNKS, 4096), DTYPE_OUT) for li in HOOK_LAYERS}
    nchunks = np.zeros(len(local), np.int16)
    ntok    = np.zeros(len(local), np.int16)
    nb = 0
    def _clean(deep=False):
        if deep: gc.collect()
        if DEV == "mps": torch.mps.empty_cache()

    def _forward(rows_):
        global GATHER
        enc = tok([fmt(local[j]) for j in rows_], return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_LEN).to(DEV)
        GATHER, nch = chunk_index(enc["attention_mask"])
        with torch.inference_mode():
            base_m(**enc, output_attentions=False, use_cache=False)
        for li in HOOK_LAYERS:
            out[li][rows_] = BUF[li].numpy()
        nchunks[rows_] = nch.numpy().astype(np.int16)
        ntok[rows_]    = enc["attention_mask"].sum(1).cpu().numpy().astype(np.int16)
        del enc; _clean(deep=(nb % DEEP_CLEAN_EVERY == 0))

    def _run(rows_, depth=0):
        nonlocal nb
        try:
            _forward(rows_); nb += 1
        except RuntimeError as e:
            if "out of memory" not in str(e).lower() or depth > 6: raise
            gc.collect()
            if DEV == "mps": torch.mps.empty_cache()
            if len(rows_) == 1:
                _clean(deep=True); time.sleep(2.0)
                try:
                    _forward(rows_); nb += 1
                except RuntimeError as e2:
                    if "out of memory" in str(e2).lower():
                        raise MemoryError(
                            "MPS is wedged -- a single row OOMed after a deep clean.\n"
                            "  Restart the kernel and re-run; finished shards are skipped.")
                    raise
            else:
                h = len(rows_) // 2
                _run(rows_[:h], depth + 1); _run(rows_[h:], depth + 1)

    t0 = time.time()
    for bt in _batches(order, local): _run(bt)
    np.savez_compressed(os.path.join(OUTDIR, f"shard_{si:04d}.npz"),
                        row_idx=np.arange(lo, hi), n_chunks=nchunks, n_tok=ntok,
                        **{f"layer_{li}": out[li] for li in HOOK_LAYERS})
    dt = time.time() - t0
    print(f"  shard {si:04d}  {len(local):4d} rows  {dt:7.1f}s  {len(local)/dt:5.2f} rows/s"
          f"  | chunks median {int(np.median(nchunks))}  short(<{MIN_CHUNKS}) {int((nchunks<MIN_CHUNKS).sum())}")
    del out; _clean(deep=True)

In [ ]:
for si in todo:
    run_shard(si)
print("\nextraction complete" if todo else "nothing to do -- all shards present")

In [ ]:
# ---- verify + manifest -----------------------------------------------------
shards = sorted(f for f in os.listdir(OUTDIR) if f.startswith("shard_"))
cov = np.concatenate([np.load(os.path.join(OUTDIR, f))["row_idx"] for f in shards]) if shards else np.array([])
missing = sorted(set(range(N)) - set(cov.tolist()))
print(f"shards {len(shards)} | rows {len(cov):,}/{N:,} | dup {len(cov)-len(set(cov.tolist()))}"
      f" | missing {len(missing)}")

if not missing:
    nch = np.zeros(N, np.int16); ntk = np.zeros(N, np.int16)
    for f in shards:
        d = np.load(os.path.join(OUTDIR, f))
        nch[d["row_idx"]] = d["n_chunks"]; ntk[d["row_idx"]] = d["n_tok"]
    df["n_chunks"], df["n_tok"] = nch, ntk
    print("\nchunks per row, by corpus:")
    print(df.groupby("corpus").n_chunks.agg(["count","median","min","max"]).to_string())
    short = df[df.n_chunks < MIN_CHUNKS]
    print(f"\nrows with <{MIN_CHUNKS} chunks: {len(short)} "
          f"({len(short)/len(df)*100:.1f}%) -- these get no position statistics")
    if len(short): print(short.groupby("corpus").size().to_string())
    trunc = (df.n_tok >= MAX_LEN).mean()*100
    print(f"rows hitting the {MAX_LEN}-token cap: {trunc:.1f}%")

    df.to_csv(os.path.join(OUTDIR, "rows.csv"), index=False)
    json.dump(dict(model=MODEL_ID, layers=HOOK_LAYERS, system_prompt=SYSTEM_PROMPT,
                   max_len=MAX_LEN, chunking="proportional",
                   min_chunk_tok=MIN_CHUNK_TOK, max_chunks=MAX_CHUNKS,
                   min_chunks=MIN_CHUNKS, rows=int(N), shard=SHARD, corpus=CORPUS,
                   dtype="float16", extract=EXTRACT, baseline_n=int(BASELINE_N)),
              open(os.path.join(OUTDIR, "manifest.json"), "w"), indent=2)
    print("\nwrote rows.csv + manifest.json -- v15c can run now")

for h in HANDLES: h.remove()
del model, base_m
gc.collect()
if DEV == "mps": torch.mps.empty_cache()
print("hooks removed, model released")